# 1.0 Load packages and data

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.impute import SimpleImputer
from scipy.stats import entropy
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from matplotlib.colors import LinearSegmentedColormap
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn_extra.cluster import KMedoids
from scipy.stats import chi2_contingency, kruskal, chisquare
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import adjusted_rand_score
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import silhouette_score, classification_report, confusion_matrix
from matplotlib.ticker import FixedLocator

In [ ]:
# Load data
df = pd.read_csv("resampled_tus.csv")

In [ ]:
# Load data
df1 = pd.read_csv("df_elect.csv")

In [ ]:
df.head()

# 2.0 Household archetypes

## 2.1 Feature engineering

One of the metrics that indicates the rigidity of a household in energy demand is the load factor. The load factor is  the ratio of daily mean to maximum peak of the day. It can also be used for any period of the day, in which case, it is defined as the ratio of the mean demand over a period of time, to the maximum peak demand at that same time.

This concept relies on the peakiness and spread of the load behaviour across a period in consideration, and measures the “peakiness” of a households load profile (Li et al., 2018; McLoughlin  et al., 2013). Ranging from 0 to 1, higher load factors indicate more consistent demand, such as households that do not have room for additional activity (high rigidity), while lower load factors are indicative of profiles with tall and narrow peaks, which indicate where households can accommodate more activities.

As a result, I will apply these to teriff rate periods. Torriti and Yunusov (2020) identifying the periods where engagements are at peaks, citing the work by Hledik et al. and the Centre for Sustainable Energy (CSE). For the Hledik and co study, these are only mornings, while the CSE shows a first peak in the morning and second in the evening. While Torriti & Yunusov used only evening peeak, our work will consider all the periods, include overnight, where there almost no activity.

Ref:
1. Matthew Li, David Allinson, Miaomiao He. 2018. Seasonal variation in household electricity demand: A comparison of monitored and synthetic daily load profiles, Energy and Buildings, Volume 179, Pages 292-300, https://doi.org/10.1016/j.enbuild.2018.09.018.

2. Fintan McLoughlin, 2013. https://arrow.tudublin.ie/cgi/viewcontent.cgi?article=1065&context=engdoc&utm

3. Torriti and Yunusov (2020). It’s only a matter of time: Flexibility, activities and time of use tariffs in the United Kingdom. Energy Research & Social Science, Vol 69, pp. 3-7, https://doi.org/10.1016/j.erss.2020.101697

In [ ]:
# Aggregate minutes per household, timebin, activity across the day
df_ = (df
    .groupby(["mainid", "starttime_of_day_min"])["bin_duration_minutes"]
    .sum()
    .reset_index())

I will pivot the dataframe so as to understand the pattern of engagement of each household.

In [ ]:
# pivot the table to index by household
df_pivot = df_.pivot_table(
    index="mainid",
    columns=["starttime_of_day_min"],
    values="bin_duration_minutes",
    fill_value=0)   # replace missing values with 0

df_pivot

## 2.1 Define time windows

I will define the different electricity tariff windows, which also reflect where people participate in energy using activities. 

In [ ]:
# Define peak times
# Infer bin size in minutes
time_bins_min = np.arange(0, 1440, 30, dtype=int) 

# morning peak run between 07:00 am and 11:00 am
day_engagement = time_bins_min[
    (time_bins_min >= 7*60) & 
    (time_bins_min < 16*60)]

# Evening peaks run between 4:00pm and 23:00pm
evening_engagement = time_bins_min[
    (time_bins_min >= 16*60) &
    (time_bins_min < 23*60)]

# Off peak run between 11am and 4pm
offpeak_engagement = time_bins_min[
    ((time_bins_min >= 23*60) &
    (time_bins_min < 24*60)) |
    ((time_bins_min >= 0*60) &
    (time_bins_min < 7*60))]

## 2.2 Compute time variabilities

Apply the defined times to the pivoted dataframe.

In [ ]:
# Compute summary features for the peaks
df_time_features = pd.DataFrame(index=df_pivot.index)

In [ ]:
# Add metrics to the data
# Get the sum of minutes within the peaks
df_time_features['morning_peak_share'] = df_pivot[day_engagement].sum(axis=1)  # morning period
df_time_features['evening_peak_share'] = df_pivot[evening_engagement].sum(axis=1)  # evening period
df_time_features['day_off_peak_share'] = df_pivot[offpeak_engagement].sum(axis=1)  # evening period

Please note that these peak shares are the total minutes spent within the window/period.

In [ ]:
# Calculate the standard deviation within each households 
df_time_features['usage_std'] = df_pivot.std(axis=1)

In [ ]:
# Get only active bins across the day
df_time_features['active_bins'] = (df_pivot > 0).sum(axis=1)

Load factor indicates how people engage in their activities, pointing to spread and demand. I will calate this for each household. It is the ratio of the mean demand per time to the maximum peak demand.

In [ ]:
# Add daily load factor
df_time_features['load_factor'] =  df_pivot.mean(axis=1)/df_pivot.max(axis=1).replace(0, np.nan)

# fill NaN with 0
df_time_features['load_factor'] = df_time_features['load_factor'].fillna(0)

Calculate the load factor for the time periods as well. This should provide us with the idea of peoples mode of engagement: momentarily or sustained at the different tariff rate periods.

In [ ]:
# Calculate for the morning period
# Highlight the times in the
daytime_cols = [col for col in df_pivot.columns if col in day_engagement]
df_time_features['daytime_lf'] = df_pivot[daytime_cols].mean(axis=1) / df_pivot[daytime_cols].max(axis=1)

# Replace infinities and NaN with the nan
df_time_features['daytime_lf'] = df_time_features['daytime_lf'].replace(
    [np.inf, -np.inf], np.nan)

# change morning_peak_lf to daytime_lf

In [ ]:
# Evening peak columns
evening_cols = [col for col in df_pivot.columns if col in evening_engagement]
df_time_features['evening_lf'] = df_pivot[evening_cols].mean(axis=1) / df_pivot[evening_cols].max(axis=1)

# Replace infinities and NaN with the nan
df_time_features['evening_lf'] = df_time_features['evening_lf'].replace(
    [np.inf, -np.inf], np.nan)

In [ ]:
# Off peak day columns
offpeak_cols = [col for col in df_pivot.columns if col in offpeak_engagement]
df_time_features['offpeak_lf'] = df_pivot[offpeak_cols].mean(axis=1) / df_pivot[offpeak_cols].max(axis=1)

# Replace infinities and NaN with the nan
df_time_features['offpeak_lf'] = df_time_features['offpeak_lf'].replace(
    [np.inf, -np.inf], np.nan)

In [ ]:
# Reset index to get mainid as column
df_time_features = df_time_features.reset_index()

# View the time features
df_time_features

## 2.3 Compute required household characteristics

These are to be merged to the time features calculations above.

In [ ]:
# Aggregate the data from the main dataset
# Group by mainid
df_hhold = df1.groupby("mainid", observed=False)

In [ ]:
# Convert to dataframe
df_household = pd.DataFrame({"mainid": df_hhold.size().index})

In [ ]:
# Select columns to be extracted
cols = ['under_age5', 'age5to11', 'age12to16', 'hhold_size_group',
        'resp_work_type', 'tenure', 'household_employment_group', 'income_category']

In [ ]:
# All the features for any household  are the same. Take the first of the most frequent
# Create a function to process this at a go
def mode(value):
    return value.value_counts().idxmax()

In [ ]:
# Extract each column using mode
# All values for each mainid should be the same
for col in cols:
    features = df_hhold[col].apply(mode)  # Get mode per mainid
    df_household[col] = df_household["mainid"].map(features)  # Map to df_household

In [ ]:
# Convert the numerical columns to numeric type
num_cols = ['under_age5', 'age5to11', 'age12to16']
for col in num_cols:
    df_household[col] = pd.to_numeric(df_household[col], errors="coerce").astype(int)

# 3.0 What are the types of households and their pattern of energy use?

To cluster the households, I will use the time features and the household features. Also, having in mind there are some households who behave differently than others, a robust model will be utilised for the modelling.

In [ ]:
# Merge time features with your household demographics
df_cluster_merge = df_time_features.merge(df_household, on='mainid', how='left')
df_cluster_merge

You will observe that the time variabilities for the periods will have Nan. And by the way, if you include the columns in the modelling, it will be clashing with the full day load factor, beacuse the day load factor also has day period, eveining period and off peak period.

## 3.1 Select clustering features

In [ ]:
# Numerical features including the demographics and time summaries
num_features = ['under_age5', 'age5to11', 'age12to16',     # Demographics
                'morning_peak_share', 'evening_peak_share', 'day_off_peak_share',
                'usage_std', 'load_factor', 'active_bins']     # Time behavioral

# Categorical features
cat_features = ['income_category', 'hhold_size_group', 'tenure', 'household_employment_group']

Recall that peak share are the minutes spent within a period.

In [ ]:
# Apply the column transformer to all the categorical features
ct = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
], remainder='drop')

In [ ]:
# Merge the transformed columns to the numerical
df_clustering = df_cluster_merge[num_features + cat_features]

In [ ]:
# Transform the data
df_clustering = ct.fit_transform(df_clustering)

## 3.2 Silhouette test

The Silhouette test allows for modelling the data with several clustering number. It compares all the clusters to understand what is the best clustering for the data. Applying the best outcomes ensures defensibility.

In [ ]:
# Container to store results for each k value
k_sil = []

# Test k values from 2 to 20 clusters
for k in range(2, 30):
    
    # Fit k-medoids clustering with k clusters
    model = KMedoids(
        n_clusters=k,           # Number of clusters to form
        metric='manhattan',     # Distance metric
        init="k-medoids++",     # Smart initialisation
        random_state=42         # Fixed seed for reproducibility
    )
    
    # Cluster the data and get cluster assignments
    label = model.fit_predict(df_clustering)
    
    # Calculate silhouette score (measures cluster separation quality)
    sil = silhouette_score(df_clustering, label, metric='manhattan')
    
    # Find smallest cluster size
    min_size = pd.Series(label).value_counts().min() # Small values indicate some clusters have very few members
    
    # Store results for this k value
    k_sil.append({
        "k": k,                      # Number of clusters
        "silhouette": sil,           # Quality score
        "min_cluster_size": min_size # Smallest cluster
    })

In [ ]:
# Sort the k values from the biggest to smallest
k_sil = pd.DataFrame(k_sil).sort_values("silhouette", ascending=False)
k_sil

The best silhouette score is k=19, showing that clustering performs best when k=19 is applied. I will use the best silhouette score for the clustering. If you increase the range, the best k will increase, but the number of households in the best k will become very small. For the sake of maintaining privacy and less granularity, a run between 2 and 20 is better, but this is open to be reviewed.

In [ ]:
# Save the best k
best_k = k_sil['k'].iloc[0]    # Use the index

## 3.3 Data Modelling

In [ ]:
# Apply the best k
model = KMedoids(
        n_clusters=best_k,      # number of clusters to form
        metric='manhattan',     # distance metric
        init="k-medoids++",     # initialisation
        random_state=42         # fixed seed for reproducibility
    )

In [ ]:
# Cluster the data and get cluster assignments
label = model.fit_predict(df_clustering)

In [ ]:
# check cluster distribution
# First convert to dataframe
df_clusters = pd.DataFrame({
    "mainid": df_pivot.index,
    "cluster": label})

In [ ]:
# Get the counts in percentages
cluster_percent = df_clusters["cluster"].value_counts(normalize=True) * 100
cluster_percent

The biggest cluster is 17.93% of the whole sample, while the least is 0.28%. The least could be removed, but since there are no identifiable features, I can leave it.

In [ ]:
# Create a bar chart showing the percentage distribution of households across clusters
ax = cluster_percent.plot(
    kind="bar",
    figsize=(18, 5),
    color="#36454F")

ax.set_xlabel("Household Cluster")  # Label the x-axis with the household cluster categories
ax.set_ylabel("Households (%)")  # Label the y-axis to show that values are percentages of households
ax.set_title("Cluster Distribution by Percentage", fontweight="bold")  # Add a bold title to describe the chart
plt.xticks(rotation=0)  # Keep x-axis labels horizontal for readability
plt.tight_layout()  # Adjust spacing so labels and title fit neatly
plt.savefig("Cluster Count.png") # Save the figure
plt.show()# Display the chart

In [ ]:
# Merge the clusters with the main dataframe, df_elect
df_elect = df1.merge(df_clusters, on='mainid', how='left')

## 3.4 Model Evaluation

Evaluation confirms how well the model did in separating the households.

### 3.4.1 Cluster Predictability

This assesses the degree to which cluster assignments could be predicted from the original feature space. We used the K-Medoid for the clustering, so applying a supervised learning algorithm to the same, we want to understand if the same patterns are learnable. I wiil train a Random Forest classifier with 100 decision trees using 5-fold cross-validation.

Random Forest is used because it
1. handles mixed data types,
2. captures non-linear relationships,
3. provides feature importance rankings, and
4. is robust to overfitting.

Five-fold cross-validation was used to obtain unbiased accuracy estimates.

In [ ]:
# Prepare data for quality assessment
X = df_clustering

In [ ]:
# Train classifier to predict clusters
rf = RandomForestClassifier(n_estimators=100, random_state=42)

In [ ]:
# Cross validate the classifier
scores = cross_val_score(rf, X, label, cv=5, scoring='accuracy')

# Print the cluster predictability
print(f"Cluster predictability: {scores.mean():.3f} ± {scores.std():.3f}")

The observed high classification accuracy indicates that cluster assignments are highly determined by the input features. This confirms that the clustering solution identified meaningful and interpretable household segments instead of arbitrary groupings.

### 3.4.2 Cluster Stability Analysis

This assesses how consistent clustering results are across different random initialisations. It tests whether the clusters represent robust, reproducible patterns or are artifacts of one particular random starting point. High stability means robust stability and meaningful clusters.

In [ ]:
# Get the number of clusters from the labels
n_clusters = len(np.unique(label))

# Store cluster labels from multiple runs
labels_runs = []

# Run k-medoids clustering 10 times with different random seeds
for seed in range(20):
    # Fit k-medoids with a different random initialisation each time
    kmed = KMedoids(
        n_clusters=n_clusters,    # Same number of clusters each time
        metric='manhattan',        # Same distance metric
        init='k-medoids++',       # Same initialisation method
        random_state=seed         # Creates variation
    ).fit(X)
    
    # Store the cluster labels from this run
    labels_runs.append(kmed.labels_)

In [ ]:
# View
labels_runs

In [ ]:
# Create an empty container
aris = []

# Compare every pair of clustering results
for i in range(len(labels_runs)):          # First clustering result
    for j in range(i+1, len(labels_runs)): # Second clustering result (avoid duplicates)
        
        # Calculate Adjusted Rand Index (ARI) between this pair
        # ARI accounts for chance agreement
        ari = adjusted_rand_score(labels_runs[i], labels_runs[j])
        aris.append(ari)  # Append to the container

In [ ]:
# View
aris

In [ ]:
# Print the statistics of the ARI
print(f"Mean ARI: {np.mean(aris):.3f} ± {np.std(aris):.3f}")
print(f"Min ARI:  {np.min(aris):.3f}")
print(f"Max ARI:  {np.max(aris):.3f}")

The ari showed that there are observed patterns, and that the clusters are moderately stable. I tried to improve this stability by increasing the number of run from 2-20 to 2-30 runs, but there is a trade off to be made. The best k when the model is run between 2 and 30 is 28, and its ARI gives better values. However, this had fewer households, with the least cluster having 13 households, which is not a granularity that we want, and in compliance with safe practices.

I stuck with k=19 because I want to ensure sufficient granularity that capture household heterogeneity, adequate cluster sizes for robust statistical analysis, acceptable stability, and interpretable, actionable segments for demand response targeting. The moderate stability reflects the inherent complexity of household electricity usage patterns and the granularity of our segmentation, rather than poor cluster quality which is evidenced by 98.9% cluster predictability.

### 3.4.3 Statistical Validation

Test if clusters differ significantly on categorical and numerical features. The tests used are Chi-square and Cramér's V for categorical features
while Kruskal-Wallis, is applied to numerical features.

In [ ]:
# Add cluster to social demography
df_cluster_merge = df_cluster_merge.merge(df_clusters, on='mainid', how='left')

In [ ]:
# Define features to test
# Numerical features: Demographics + Behavioral
num_cols = ['under_age5', 'age5to11', 'age12to16',     # Demographics
            'morning_peak_share', 'evening_peak_share', 'usage_std',     # Behavioral (electricity usage patterns)
            'load_factor', 'active_bins']

# Categorical features
cat_cols = ['income_category', 'hhold_size_group', 'resp_work_type', 'tenure',  # Socio-economic characteristics
            'household_employment_group']

In [ ]:
# Create function for Chi-square test + Cramér's V
def calculate_association(df, cluster_col, feature_col):
    # Generate a contingency table for cluster membership and the categorical feature
    table = pd.crosstab(df[cluster_col], df[feature_col])
    
    # Test for association using the chi-square test of independence
    chi2, p_value, dof, expected = chi2_contingency(table)
    
    # Estimate association strength using Cramér's V
    total = table.sum().sum()  # total number of observations
    row, column = table.shape    # number of rows and columns in the contingency table
    cramers_v = np.sqrt(chi2 / (total * (min(row, column) - 1)))

    # Return the p-value, effect size (Cramér's V), and chi-square statistic
    return p_value, cramers_v, chi2

In [ ]:
# Initialise a list to collect chi-square association results for categorical features
cat_results = []

# Test the association between cluster membership and each categorical variable
for col in cat_cols:
    p_value, cramers_v, chi2 = calculate_association(df_cluster_merge, "cluster", col)
    
    # Classify statistical significance using the p-value threshold
    if p_value < 0.001:
        sig_text = "Highly significant (p < 0.001)"
    elif p_value < 0.05:
        sig_text = "Significant (p < 0.05)"
    else:
        sig_text = "Not significant (p ≥ 0.05)"
    
    # Classify effect size based on Cramér's V
    if cramers_v < 0.1:
        strength = "Negligible"
    elif cramers_v < 0.3:
        strength = "Weak"
    elif cramers_v < 0.5:
        strength = "Moderate"
    else:
        strength = "Strong"

    # Append both numerical results and interpretive labels
    cat_results.append({
        "Feature": col,
        "Chi-square": round(chi2, 2),
        "p-value": round(p_value, 4),
        "Significance": sig_text,
        "Cramers_V": round(cramers_v, 4),
        "Effect_Size": strength
    })

In [ ]:
# Convert results into a summary DataFrame
cat_summary = pd.DataFrame(cat_results)

# View the summary table
cat_summary

All the variables are statistically significant with the clusters (p < 0.001 across board). The strength of association (Cramér’s V) is strongest for household size and household employment group, followed by income and tenure, and still meaningful (moderate‑to‑strong) for respondent work type. This suggests that clusters are most distinctly characterised by household size and household employment structure, with tenure and income also playing large roles. Work type still matters, just a bit less than the others.

In [ ]:
# Create an empty list to store Kruskal-Wallis test results for numerical variables
num_results = []

# Loop through each numerical column in the dataset
for col in num_cols:
    # Split the current numerical feature into groups based on cluster membership
    # and remove missing values from each group
    groups = [df_cluster_merge[df_cluster_merge["cluster"] == c][col].dropna()
              for c in sorted(df_cluster_merge["cluster"].unique())]
    
    # Perform the Kruskal-Wallis test to check whether the distribution
    # of the numerical feature differs significantly across clusters
    stat, p_value = kruskal(*groups)
    
    # Interpret statistical significance based on the p-value
    if p_value < 0.001:
        sig_text = "Highly significant (p < 0.001)"
    elif p_value < 0.01:
        sig_text = "Very significant (p < 0.01)"
    elif p_value < 0.05:
        sig_text = "Significant (p < 0.05)"
    else:
        sig_text = "Not significant (p ≥ 0.05)"

    # Store the feature name, test statistic, p-value,
    # and significance interpretation in the results list
    num_results.append({
        "Feature": col,
        "H-statistic": round(stat, 2),
        "p-value": round(p_value, 4),
        "Significance": sig_text
    })

In [ ]:
# Convert the list of results into a DataFrame for easier interpretation
num_summary = pd.DataFrame(num_results)

# Display the summary table
num_summary

Kruskal–Wallis tests revealed significant differences across clusters for all numerical household composition variables (p < 10⁻¹⁸). This indicate that clusters have systematic variation in the number and age distribution of children, this provides strong evidence that the derived clusters reflect meaningful differences in household structure when combined with categorical variable analyses.

### 3.4.4 Does the distribution of each socio-demographic characteristic within a given cluster differ from the distribution expected based on the overall sample?

In [ ]:
# Function for Chi-square goodness-of-fit test
def calculate_goodness_of_fit(df, cluster_col, feature_col, cluster_value):

    # Remove missing values for the feature being tested
    feature_df = df[[cluster_col, feature_col]].dropna()

    # Calculate the overall distribution of the feature
    overall_counts = feature_df[feature_col].value_counts().sort_index()

    # Convert overall counts to proportions
    overall_proportions = overall_counts / overall_counts.sum()

    # Get observations belonging to the selected cluster
    cluster_data = feature_df[
        feature_df[cluster_col] == cluster_value
    ]

    # Count each category within the cluster
    observed = (
        cluster_data[feature_col]
        .value_counts()
        .reindex(overall_counts.index, fill_value=0)
        .sort_index()
    )

    # Calculate expected counts using the overall sample distribution
    expected = overall_proportions * observed.sum()

    # Chi-square goodness-of-fit test
    chi2, p_value = chisquare(
        f_obs=observed,
        f_exp=expected
    )

    return chi2, p_value, observed, expected

In [ ]:
# Initialise list for results
cat_results = []

# Loop through each categorical feature
for col in cat_cols:

    # Loop through each cluster
    for cluster in sorted(df_cluster_merge['cluster'].dropna().unique()):

        chi2, p_value, observed, expected = calculate_goodness_of_fit(
            df_cluster_merge,
            'cluster',
            col,
            cluster
        )

        # Statistical significance
        if p_value < 0.001:
            sig_text = 'Highly significant (p < 0.001)'
        elif p_value < 0.05:
            sig_text = 'Significant (p < 0.05)'
        else:
            sig_text = 'Not significant (p ≥ 0.05)'

        # Store results
        cat_results.append({
            'Cluster': cluster,
            'Feature': col,
            'Chi-square': round(chi2, 2),
            'p-value': p_value,
            'Significance': sig_text
        })


# Convert results to DataFrame
cat_summary = pd.DataFrame(cat_results)

cat_summary

# 4.0 What is the characteristics of each household?

## 4.1 Descriptive analysis

In [ ]:
# Add cluster to social demography
df_household = df_household.merge(df_clusters, on='mainid', how='left')

In [ ]:
# Compute descriptive statistic for the numeric components
# Select the columns
num_cols = ['under_age5', 'age5to11', 'age12to16']

# Calculate the central tencies and round to 2 decimal
num_summary = df_household.groupby('cluster')[num_cols].agg(['mean', 'median', 'std']).round(2)

# Add overall mean as reference row
overall = df_household[num_cols].agg(['mean', 'median', 'std']).round(2).T.stack()
print("Overall reference:")
print(df_household[num_cols].agg(['mean', 'median', 'std']).round(2))

num_summary

Adult-only household clusters include Clusters 1, 6, 9, 13 without (children). Clusters 0, 5, 7, 10, 17 are mostly adult households, with  occasional single young child while Clusters 4, 14, 18 are mixed family households with moderate children.

Clusters 2, 3, 8, 11, 12, 15, 16 are Strong family clusters with about 1–3 children commonly present and Cluster 11 are households where nearly all households have a school-age child, suggesting a very strong behavioural signal.

In [ ]:
# Specify columns
num_cols = ['under_age5', 'age5to11', 'age12to16']

# Loop through each numerical column in the list
for col in num_cols:
    # Print the column name as a heading
    print(f"\n{col}")

    # For the current column, calculate the percentage distribution of values within each cluster:
    df_age = (
        df_household.groupby("cluster")[col]
        .value_counts(normalize=True)# group by cluster and normalise to proportions within each cluster
        .mul(100)   # count the frequency of each value and convert to percentages
        .round(1)    # round to 1 decimal place
        .unstack()  # reshape so clusters are rows and values are columns
        .fillna(0)  # replace missing combinations with 0
    )
    print(df_age)   # Display the result from the last column processed in the loop

Cluster 0: 3.8% have children under 5, 7.5% have children aged 5–11, and 3.8% have children aged 12–16.

Cluster 1: 22.8% have children under 5, 31.7% have children aged 5–11, and 26.3% have children aged 12–16.

Cluster 2: 0.6% have children under 5, 0% have children aged 5–11, and 0% have children aged 12–16.

Cluster 3: 13.2% have children under 5, 16.2% have children aged 5–11, and 17.6% have children aged 12–16.

Cluster 4: 0% have children under 5, 0% have children aged 5–11, and 2.3% have children aged 12–16.

Cluster 5: 3.5% have children under 5, 5.3% have children aged 5–11, and 0% have children aged 12–16.

Cluster 6: 0% have children under 5, 1.9% have children aged 5–11, and 3.8% have children aged 12–16.

Cluster 7: 23.0% have children under 5, 29.2% have children aged 5–11, and 31.9% have children aged 12–16.

Cluster 8: 10.6% have children under 5, 63.2% have children aged 5–11, and 31.6% have children aged 12–16.

Cluster 9: 1.6% have children under 5, 0% have children aged 5–11, and 1.6% have children aged 12–16.

Cluster 10: 0% have children under 5, 1.8% have children aged 5–11, and 0% have children aged 12–16.

Cluster 11: 1.6% have children under 5, 0% have children aged 5–11, and 0% have children aged 12–16.

Cluster 12: 37.1% have children under 5, 40.3% have children aged 5–11, and 16.1% have children aged 12–16.

Cluster 13: 33.3% have children under 5, 39.2% have children aged 5–11, and 37.3% have children aged 12–16.

Cluster 14: 3.6% have children under 5, 5.5% have children aged 5–11, and 1.8% have children aged 12–16.

Cluster 15: 32.4% have children under 5, 50.0% have children aged 5–11, and 32.4% have children aged 12–16.

Cluster 16: 36.7% have children under 5, 42.9% have children aged 5–11, and 26.5% have children aged 12–16.

Cluster 17: 0% have children under 5, 0% have children aged 5–11, and 0% have children aged 12–16.

Cluster 18: 1.9% have children under 5, 3.7% have children aged 5–11, and 1.9% have children aged 12–16.

Cluster 19: 0% have children under 5, 2.9% have children aged 5–11, and 0% have children aged 12–16.

Cluster 20: 16.0% have children under 5, 28.0% have children aged 5–11, and 24.0% have children aged 12–16.

Cluster 21: 0% have children under 5, 0% have children aged 5–11, and 0% have children aged 12–16.

Cluster 22: 0.7% have children under 5, 0% have children aged 5–11, and 0% have children aged 12–16.

Cluster 23: 23.8% have children under 5, 37.6% have children aged 5–11, and 36.6% have children aged 12–16.

Cluster 24: 0% have children under 5, 0% have children aged 5–11, and 0% have children aged 12–16.

Cluster 25: 0% have children under 5, 0% have children aged 5–11, and 0% have children aged 12–16.

Cluster 26: 0% have children under 5, 0% have children aged 5–11, and 0% have children aged 12–16.

Cluster 27: 0% have children under 5, 0% have children aged 5–11, and 0% have children aged 12–16.

Cluster 28:0% have children under 5, 0% have children aged 5–11, and 0% have children aged 12–16.

Create a binary indicator showing whether a household has any young children

In [ ]:
# A household is marked True if it has at least one child in any of the age groups:
# under 5, 5 to 11, or 12 to 16
df_household["is_young"] = (
    (df_household["under_age5"] > 0) |
    (df_household["age5to11"] > 0) |
    (df_household["age12to16"] > 0)
)

In [ ]:
# Convert the boolean values in `is_young` to readable category labels
df_household["is_young"] = df_household["is_young"].map({
    True: "Has Children",  # True becomes 'Has Children'
    False: "No Children"   # False becomes 'No Children'
})

In [ ]:
# Select the necessary columns
cat_cols = ['hhold_size_group', 'is_young', 'income_category', 'resp_work_type', 'household_employment_group', 'tenure']

# Calculate the percentage ratio within each cluster
for col in cat_cols:
    print(f"\n{col} by Cluster:")
    print(pd.crosstab(df_household['cluster'], df_household[col], normalize='index') * 100)
    print("--------------------------------------------------------")

In [ ]:
# Compute the pecentage into one table
cat_summary = pd.concat([
    pd.crosstab(df_household['cluster'], df_household[col], normalize='index')
    .mul(100).round(1)
    .add_prefix(f"{col}_")
    for col in cat_cols
], axis=1)

In [ ]:
# Save the table
cat_summary.to_csv('in_cluster_composition.csv', index=True)

In [ ]:
cat_summary

Summary

Cluster 0
This cluster is overwhelmingly composed of small households (95.6%), with very few households with children (12%). It is predominantly low income (80.5%), with most respondents onsite workers, and households are typically partly employed. Housing tenure is mixed but dominated by mortgage holders, alongside notable private and council renting, indicating economically constrained working households.

Cluster 1
Cluster 1 is mainly medium‑sized households (97.6%) with a high prevalence of children (64%), making it a strongly family‑oriented cluster. Income is mixed, with representation across high, middle, and upper‑middle incomes, and employment is dominated by fully‑employed households. Most households are mortgage holders, reflecting stable, working family homeowners.

Cluster 2
This cluster consists almost entirely of small households (98%), with almost no children present. It is overwhelmingly low income, and strongly characterised by retired households, living primarily in owner‑occupied housing. This cluster represents low‑income retired homeowners, likely older households with minimal household activity variation.

Cluster 3
Cluster 3 is dominated by medium‑sized households (98.5%) with a moderate presence of children (40%). The cluster is largely low income, with households mostly partly employed or onsite workers, and tenure is almost exclusively owner‑occupied. This reflects working‑age families in stable ownership but with constrained incomes.

Cluster 4
This cluster is made up entirely of small households (100%), with virtually no children. It consists almost exclusively of middle‑income homeowners, with a high representation of retired households. The profile is that of older, child‑free, middle‑income owner‑occupiers with stable housing conditions.

Cluster 5
Cluster 5 features small households (98%) with very low child presence. Income is predominantly low, and employment is characterised by partly unemployed households. Tenure is mixed between private renting, council renting, and third‑party ownership, suggesting economically vulnerable renters with unstable labour attachment.

Cluster 6
This cluster contains almost exclusively small households, no children, and is concentrated in the lower‑middle income category. Employment is dominated by retired households, and homes are overwhelmingly owner‑occupied. It represents lower‑middle income retired homeowners with stable housing but limited income.

Cluster 7
Cluster 7 consists primarily of medium‑sized households (98%) and has a very high prevalence of children (71%). Income is very high, employment is partly employed, and tenure is dominated by mortgages. This cluster represents affluent, working family households with children.

Cluster 8
This cluster contains mostly medium households with the highest child presence (84%) among all clusters. Income is concentrated in upper‑middle income, with most respondents working onsite or remotely, and tenure is largely third‑party owned, pointing to corporate or institutional housing among affluent families.

Cluster 9
Cluster 9 consists entirely of small households, almost entirely child‑free, and is composed of very high income households. Employment is split between onsite and remote work, and tenure is dominated by mortgages, indicating affluent, child‑free professional households.

Cluster 10
This cluster is dominated by small households, no children, middle income, and is heavily retired. Housing tenure is overwhelmingly owner‑occupied, representing stable, retired middle‑income homeowners.

Cluster 11
Cluster 11 is composed almost entirely of small households, child‑free, in the lower‑middle income bracket, and dominated by retired households. Homeownership is very high, indicating older, financially modest owner‑occupiers.

Cluster 12
This cluster consists mainly of medium‑sized households with a very high prevalence of children (73%). Income is concentrated in middle income, employment is largely partly employed, and tenure is split between mortgages and ownership. It represents middle‑income family households with children.

Cluster 13
Cluster 13 is similar to Cluster 12 but with an even higher child presence (76%). Households are medium‑sized, lower‑middle income, and primarily employed onsite. Ownership and mortgages dominate tenure, marking working families with moderate incomes.

Cluster 14
This cluster is predominantly small households, child‑free, and exclusively upper‑middle income. Employment is mainly partly employed, with housing split between mortgages and ownership, indicating affluent, child‑free households.

Cluster 15
Cluster 15 contains mostly medium‑sized households, a high proportion of children (68%), and is overwhelmingly low income. Employment is characterised by partly unemployed households, and tenure is dominated by private and council renting, reflecting large low‑income family renters.

Cluster 16
This cluster is entirely medium‑sized households, with a very high child presence (80%). Income is entirely upper‑middle income, employment is partly employed, and tenure is mainly mortgage‑based, representing financially stable, working families.

Cluster 17
Cluster 17 consists entirely of small, child‑free households, overwhelmingly very high income, and predominantly retired owner‑occupiers. This cluster represents wealthy retirement households.

Cluster 18
This cluster is entirely small households, mostly child‑free, in the lower‑middle income bracket. Employment is mostly onsite, housing is dominated by mortgages and private renting, suggesting older working households in modest circumstances.

Cluster 19
Cluster 19 comprises mainly small households, child‑free, and upper‑middle income, with a high share of retired households living in owner‑occupied housing.

Cluster 20
Cluster 20 is formed entirely of medium‑sized households, with moderate child presence (56%), very high income, and a mix of onsite and remote employment. Housing is overwhelmingly owner‑occupied, typifying affluent family households.

Cluster 21
Cluster 21 consists of small, child‑free households, with very high income and predominantly remote workers, reflecting high‑income professional households.

Cluster 22
This cluster contains only small households, child‑free, overwhelmingly low income, primarily onsite employed, and almost entirely owner‑occupied, suggesting low‑income owner households without children.

Cluster 23
Cluster 23 is composed of medium‑sized households, a high proportion of children (76%), low income, and primarily onsite employed, with tenure dominated by mortgages and private renting. This is a classic low‑income working family cluster.

Cluster 24
Clusters 24–28 are entirely small, child‑free households, with very strong income stratification:

Cluster 24: Very high income, owner‑occupied
Cluster 25: Low‑income, retired, third‑party housing
Cluster 26: Low‑income retired owner‑occupiers
Cluster 27: Middle‑income onsite workers, homeowners
Cluster 28: High‑income owner‑occupiers

These clusters represent non‑family, economically segmented adult households.

## 4.2 How representative are the socioeconomic groups within the different housdhold groups?

The representation‑ratio analysis reveals how common each category is within each cluster, against how common it is in the full dataset. It identifies clear socio‑economic specialisations, showing which features are over represented, and underrepresented.. 

In [ ]:
# Define a function to calculate the representation ratio
def representation_ratio(df, var):
    
    # Calculate the proportion of each category within each cluster
    within = pd.crosstab(df["cluster"], df[var], normalize="index")
    
    # Calculate the overall proportion of each category in the full dataset
    overall = df[var].value_counts(normalize=True)
    
    # Divide the within-cluster proportions by the overall proportions
    return (within / overall).round(2)

In [ ]:
# Compute representation ratios for children group
rr_is_young = representation_ratio(df_household, "is_young")

# Compute representation ratios for income category
rr_income = representation_ratio(df_household, "income_category")

# Compute representation ratios for housing tenure
rr_housing = representation_ratio(df_household, "tenure")

# Compute representation ratios for respondent work type
rr_worktype = representation_ratio(df_household, "resp_work_type")

# Compute representation ratios for household employment group
rr_working = representation_ratio(df_household, "household_employment_group")

# Compute representation ratios for household size group
rr_hh_size = representation_ratio(df_household, "hhold_size_group")

In [ ]:
# Display the representation ratio tables
rr_is_young, rr_income, rr_housing, rr_worktype, rr_working, rr_hh_size

In [ ]:
# Combine the tables
rr_combined = pd.concat([
    rr_is_young.add_prefix("is_young_"),
    rr_income.add_prefix("income_"),
    rr_hh_size.add_prefix("hh_size_"),
    rr_worktype.add_prefix("worktype_"),
    rr_working.add_prefix("employment_"),
    rr_housing.add_prefix("tenure_")
], axis=1)

# Save the tables
rr_combined.to_csv('cluster_representation_ratios.csv')

In [ ]:
# Convert to percentages
rr_combined_pct = (rr_combined - 1).mul(100).round(1)
rr_combined_pct

In [ ]:
# Save the percentages
rr_combined_pct.to_csv('Representation Ratios in Percentage.csv')

Cluster 1

Households with children: +126% over‑represented
Medium households: +152% over‑represented
High income: +279% over‑represented
Upper‑middle income: +187% over‑represented
Fully employed households: +501% over‑represented
Mortgage owners: +102% over‑represented

Interpretation: Affluent, fully employed family households with strong housing stability.

Cluster 2

Households with children: −98% under‑represented
Small households: +63% over‑represented
Low income: +135% over‑represented
Fully retired: +443% over‑represented
Owner‑occupation: +282% over‑represented

Interpretation: Low‑income retired homeowners, strongly non‑family.

Cluster 3

Households with children: +40% over‑represented
Medium households: +154% over‑represented
Low income: +123% over‑represented
Partly employed households: +558% over‑represented

Interpretation: Working‑age, lower‑income families with partial employment.

Cluster 4

Child‑free households: +36% over‑represented
Middle income: +689% over‑represented
Fully retired: +687% over‑represented
Owner‑occupation: +122% over‑represented

Interpretation: Middle‑income retired owner‑occupiers without children.

Cluster 5

Low income: +117% over‑represented
Unemployed households: +1 047% over‑represented
Renting (private & council): +206–449% over‑represented
Third‑party housing: +263% over‑represented

Interpretation: Highly vulnerable renter households with severe labour‑market exclusion.

Cluster 6

Lower‑middle income: +649% over‑represented
Fully retired households: +733% over‑represented
Owner‑occupation: +122% over‑represented

Interpretation: Lower‑middle‑income retired homeowners, economically stable in tenure.

Cluster 7

Households with children: +149% over‑represented
Very‑high income: +511% over‑represented
Medium households: +154% over‑represented
Remote workers: +115% over‑represented
Mortgage owners: +114% over‑represented

Interpretation: Very affluent, professional family households.

Cluster 8

Households with children: +197% over‑represented
Large households: +310% over‑represented
Upper‑middle income: +698% over‑represented
Third‑party housing: +1 118% over‑represented

Interpretation: Affluent large families in institutional or non‑standard tenure.

Cluster 9

Child‑free households: +35% over‑represented
Very‑high income: +511% over‑represented
Remote working: +167% over‑represented

Interpretation: High‑income, child‑free professional households.

Cluster 10

Middle income: +689% over‑represented
Fully retired households: +436% over‑represented
Owner‑occupation: +182% over‑represented

Interpretation: Retired, middle‑income homeowners.

Cluster 11

Lower‑middle income: +649% over‑represented
Fully retired households: +436% over‑represented

Interpretation: Modest‑income retired owner households.

Cluster 12

Households with children: +156% over‑represented
Medium households: +154% over‑represented
Middle income: +689% over‑represented

Interpretation: Middle‑income working families.

Cluster 13

Households with children: +169% over‑represented
Lower‑middle income: +649% over‑represented
Medium households: +148% over‑represented

Interpretation: Lower‑middle‑income working families.

Cluster 14

Upper‑middle income: +847% over‑represented
Small households: +61% over‑represented
Child‑free households: +29% over‑represented

Interpretation: Affluent, non‑family households.

Cluster 15

Households with children: +138% over‑represented
Large households: +358% over‑represented
Low income: +145% over‑represented
Partly unemployed households: +1 216% over‑represented
Council / private renting: +464–553% over‑represented

Interpretation: Large, low‑income family renters under heavy economic stress.

Cluster 16

Households with children: +180% over‑represented
Upper‑middle income: +847% over‑represented
Medium households: +158% over‑represented

Interpretation: Financially stable, affluent family households.

Cluster 17

Very‑high income: +511% over‑represented
Fully retired households: +306% over‑represented
Child‑free households: +40% over‑represented

Interpretation: Wealthy retirees.

Cluster 18

Lower‑middle income: +649% over‑represented
Partly employed households: +52% over‑represented
Private renting: +122% over‑represented

Interpretation: Transitional older households with mixed tenure and work status.

Cluster 19

Upper‑middle income: +847% over‑represented
Fully retired households: +230% over‑represented

Interpretation: Comfortable, child‑free retired households.

Cluster 20

Households with children: +97% over‑represented
Very‑high income: +742% over‑represented
Remote work: +742% over‑represented

Interpretation: Affluent, technology‑enabled family households.

Cluster 21

Very‑high income: +333% over‑represented
Remote work: +268% over‑represented
Private renting: +1 207% over‑represented

Interpretation: Mobile, high‑income remote professionals.

Cluster 22

Low income: +152% over‑represented
Onsite workers: +77% over‑represented

Interpretation: Low‑income working households, largely child‑free.

Cluster 23

Households with children: +169% over‑represented
Low income: +152% over‑represented
Renting: +591% (private) over‑represented

Interpretation: Low‑income working families in rental housing.

Cluster 24

Very‑high income: +527% over‑represented
Owner‑occupation: +200% over‑represented

Interpretation: Highly affluent, child‑free homeowners.

Cluster 25

Fully retired households: +823% over‑represented
Third‑party housing: +1 118% over‑represented

Interpretation: Economically vulnerable elderly households in institutional housing.

Cluster 26

Fully retired households: +944% over‑represented
Low income: +144% over‑represented

Interpretation: Very low‑income retirees with secure tenure.

Cluster 27

Medium households: +58% over‑represented
Onsite workers: +39% over‑represented

Interpretation: Working, non‑family households with moderate stability.

Cluster 28

High income: +1 246% over‑represented
Owner‑occupation: +200% over‑represented
Child‑free households: +40% over‑represented

Interpretation: Elite, non‑family owner‑occupier households.

### 4.2.1 Visualise representation ratio

I will plot the representation ratios as diverging bars for each cluster. The values will be converted to percentage difference from 1. For values above 0 mean, the category is over-represented in the cluster, while those below 0 mean are under-represented in the cluster. 0 means the category is represented exactly as expected.

In [ ]:
# Visualise representation ratios as separate diverging bar charts
def plot_representation_bars(rr_df, feature_name):

    # Convert representation ratio to percentage deviation from 1
    rr_percent = (rr_df - 1) * 100

    # Loop through each cluster and create a separate figure
    for cluster in rr_percent.index:
        values = rr_percent.loc[cluster]

        # Colour bars by direction of representation
        colors = [
            '#2A9D8F' if v > 0 else '#B0B0B0' if v == 0 else '#E76F51'
            for v in values
        ]  # Green = over-represented, red = under-represented, grey = exactly expected

        # Create a new figure for this cluster
        plt.figure(figsize=(12, 4))

        # Draw the bars for this cluster
        plt.bar(values.index, values.values, color=colors)

        # Add a horizontal reference line at 0
        plt.axhline(0, color='black', linewidth=1)

        # Add chart title
        plt.title(
            f'Representation Ratio of {feature_name} for Cluster {cluster}',
            fontweight='bold',
            fontsize=12
        )

        # Label axes
        plt.xlabel(feature_name, fontsize=11)
        plt.ylabel('% Over/Under-Represented', fontsize=11)

        # Add a light horizontal grid
        plt.grid(True, alpha=0.3, axis='y')

        # Keep category labels horizontal for readability
        plt.xticks(rotation=0)

        # Adjust layout so titles and labels do not overlap
        plt.tight_layout()

        # Save each cluster chart separately
        plt.savefig(
            f'Representation Ratio of {feature_name} - Cluster {cluster}.png'.replace('/', '_'),
            dpi=300,
            bbox_inches='tight'
        )

        # Display the chart
        plt.show()

        # Close the figure before the next loop iteration
        plt.close()

In [ ]:
plot_representation_bars(rr_is_young, "Children Presence")

In [ ]:
# Visualise for income
plot_representation_bars(rr_income, 'Income Category')

In [ ]:
# Visualise for household sizes
plot_representation_bars(rr_hh_size, 'Household Size')

In [ ]:
# Visualise for housing tenure
plot_representation_bars(rr_housing, 'Housing Tenure')

In [ ]:
# Visualise for respondents' work type
plot_representation_bars(rr_worktype, 'Work Type')

In [ ]:
# Visualise for household employment grouping
plot_representation_bars(rr_working, 'Employment Group')

Summary

Several clusters exhibit pronounced over‑representation of low‑income and economically constrained households, most notably clusters 0, 2, 3, 5, 15, 22, 23, 25, and 26, where low‑income representation ratios exceed unity by substantial margins. These clusters also tend to show elevated representation of non‑standard tenure types, including council housing, private renting, and third‑party ownership, pointing to higher levels of housing insecurity and economic vulnerability. In contrast, clusters 7, 8, 9, 14, 16, 17, 20, 21, 24, and 28 display strong over‑representation of upper‑middle, high, or very‑high income households, often accompanied by dominant mortgage ownership or outright homeownership, signalling greater financial stability and higher capacity for resource‑intensive lifestyles.

Employment patterns reinforce this socio‑economic divide. Fully retired households are heavily over‑represented in clusters 2, 4, 6, 10, 11, 17, 18, 19, 25, and 26, aligning closely with clusters dominated by small, child‑free households and high rates of owner‑occupation. Conversely, fully employed households are most strongly over‑represented in clusters 1, 7, 12, 13, 16, and 20, which are also characterised by higher income levels and mortgage tenure. Clusters 5 and 15 stand out for extreme over‑representation of unemployment and partial unemployment, marking them as the most labour‑market‑precarious segments.

Household composition further differentiates clusters. Family households with children are strongly over‑represented in clusters 1, 7, 8, 12, 13, 15, 16, 20, and 23, which also tend to show substantial over‑representation of medium or large household sizes. In contrast, clusters 0, 2, 4, 6, 9, 10, 11, 17, 18, 19, 24, and 28 are dominated by small, child‑free households, frequently coinciding with retirement or high‑income professional profiles.
Finally, work location patterns highlight behavioural differentiation: onsite work is over‑represented in many lower‑income and family‑oriented clusters (e.g. 0, 3, 12, 13, 22, and 23), suggesting more rigid daytime absence patterns, while remote working is disproportionately common in several high‑income clusters such as 7, 8, 9, 14, 17, 20, and 21, pointing to greater temporal flexibility and potential differences in daily energy‑use rhythms.

# 5.0 Which household groups are flexible?

## 5.1 Compute load factor per cluster

In [ ]:
# Ensure keys have the same dtype first
clusters_keyed = df_clusters.set_index('mainid')['cluster']  # Series indexed by mainid
df_cluster_merge['cluster'] = df_cluster_merge['mainid'].map(clusters_keyed)

In [ ]:
# Add the off peak
df_cluster_merge[['daytime_lf', 'evening_lf','offpeak_lf']] = df_time_features[
    ['daytime_lf', 'evening_lf','offpeak_lf']].copy()

In [ ]:
# Select load factors for visualisation
df_rigidity = df_cluster_merge[['mainid','load_factor', 'daytime_lf', 'evening_lf','offpeak_lf', 'cluster']]
df_rigidity

In [ ]:
# Compute the mean of load factors
summary_mean = df_rigidity.groupby('cluster').agg({
    'load_factor': 'mean',
    'daytime_lf': 'mean',
    'evening_lf': 'mean',
    'offpeak_lf': 'mean'
}).round(3).reset_index()

In [ ]:
# Compute the standard deviation for the load factors
summary_std = df_rigidity.groupby('cluster').agg({
    'load_factor': 'std',
    'daytime_lf': 'std',
    'evening_lf': 'std',
    'offpeak_lf': 'std'
}).round(3).reset_index()

# Rename std columns
summary_std = summary_std.rename(columns={
    'load_factor': 'load_factor_std',
    'daytime_lf': 'daytime_lf_std',
    'evening_lf': 'evening_lf_std',
    'offpeak_lf': 'offpeak_lf_std'})

In [ ]:
# Merge the mean summary and the standard deviation summary
summary_windows = summary_mean.merge(summary_std, on='cluster')
summary_windows = summary_windows.sort_values(by='cluster')

# View the data
summary_windows

In [ ]:
# Save the data
summary_windows.to_csv('Flexibility Measure Table.csv')

Across the day, the majority of clusters exhibit moderate rigidity, with load‑factor values concentrated between 0.130 and 0.199, indicating relatively stable but not fully inflexible daily activity patterns. A smaller subset of clusters displays rigid behaviour (load factors ≥ 0.200), reflecting more strongly constrained and habitual activity timing throughout the day. These rigid clusters are primarily associated with retirement or limited labour‑market engagement. In contrast, only a few clusters fall into the less rigid category (load factor < 0.130), suggesting greater temporal flexibility in daily activities. Overall, across‑the‑day rigidity is dominated by moderately rigid patterns, with limited representation of highly flexible households and a clearly identifiable group of more rigid clusters.

In [ ]:
# Visualise the load factor per cluster
# Create a new figure with a width of 10 inches and height of 4 inches
plt.figure(figsize=(10,4))

# Use the summary_windows dataframe
sns.barplot(data=summary_windows, x='cluster', y='load_factor',
            color='#AFBB77') # color sets all bars to a custom green shade

plt.xlabel('Household Cluster') # Label the x-axis
plt.ylabel('Flexibility Measure')  # Label the y-axis
plt.title("Average Flexibility Measures of Households", fontweight="bold") # Add a title to the chart and make it bold
plt.tight_layout()  # Adjust spacing so labels and title fit neatly within the figure
plt.savefig("Average Flexibility Measures of Households.png")  # Save the figure as a png image file
plt.show()  # Display the chart

In [ ]:
# Select the columns needed for plotting
df_plot = summary_windows[['cluster', 'load_factor', 'daytime_lf', 'evening_lf', 'offpeak_lf']]

# Set cluster as the x-axis
df_plot = df_plot.set_index('cluster')

In [ ]:
# Create a grouped bar chart with custom bar colors
df_plot.plot(kind='bar', figsize=(15, 4),
    color=['#A0AB75', '#2F5D8A', '#D2AB4B', '#6f4685'])

plt.xlabel('Household Cluster')  # Label the x-axis
plt.ylabel('Flexibility Measure')  # Label the y-axis
plt.title('Flexibility Measures by Household Cluster', fontweight='bold') # Add a bold title
plt.legend(['All Day', 'Daytime', 'Evening', 'Offpeak'],
          bbox_to_anchor=(1.02, 1), loc='best') # Add legend with clearer labels
plt.xticks(rotation=0)
plt.tight_layout()  # Adjust spacing
plt.show()  # Show the plot

Evenings are the most rigid time for all the household cluster groups. The daytime and daily measure have almost similar pattern, while offpeak is the least. Groups 9 and 13 also maintained high rigidity during the daytime and evening periods, but not the offpeak periods.

With all the features identified, we can now name these cluters according to their flexibility and features. The rigidity aspects were classified according to the load factor ranges:
1. Less rigid: < 0.130
2. Moderately rigid: < 0.20 
3. Rigid: <0.4
4. Very rigid: < 0.6
5. Highly rigid: < 0.8
6. Extremely rigid: >= above. 

In [ ]:
# Rename the cluster with the characteristics
cluster_names = {
    0:  "Moderately Rigid No-Children Owner-Workers",
    1:  "Moderately Rigid Working Families With Mortgages",
    2:  "Rigid Retired Small-Household Owners",
    3:  "Moderately Rigid Upper-Middle-Income Large Households",
    4:  "Moderately Rigid Very-High-Income Family Homeowners",
    5:  "Moderately Rigid Unemployed Large Private Renters",
    6:  "Rigid Older Mixed-Employment Homeowners",
    7:  "Less Rigid Lower-Middle-Income Onsite Workers",
    8:  "Less Rigid Upper-Middle-Income Parental-Leave Families",
    9:  "Moderately Rigid Retired Home-Owning Small Households",
    10: "Rigid Middle-Income Small-Household Workers",
    11: "Moderately Rigid Very-High-Income Remote/Onsite Professionals",
    12: "Less Rigid Lower-Middle-Income Retired Homeowners",
    13: "Moderately Rigid Lower-Middle-Income Retired Owners",
    14: "Moderately Rigid Shared-Ownership Family Households",
    15: "Moderately Rigid Low-Income Unemployed Council Renters",
    16: "Moderately Rigid Lower-Middle-Income Council-Renting Families",
    17: "Moderately Rigid Low-Income Medium-Household Workers",
    18: "Moderately Rigid Large Low-Income Unemployed Renters",
    19: "Moderately Rigid Lower-Middle-Income Retired Renters",
    20: "Moderately Rigid High-Income Dual-Earner Family Households",
    21: "Moderately Rigid Very-High-Income Remote-Working Renters",
    22: "Moderately Rigid Low-Income Onsite-Working Owners",
    23: "Moderately Rigid Low-Income Working Family Renters",
    24: "Moderately Rigid Very-High-Income Owner-Occupiers",
    25: "Rigid Low-Income Retired Third-Party Housing",
    26: "Rigid Low-Income Fully Retired Owners",
    27: "Moderately Rigid Middle-Income Onsite-Working Households",
    28: "Moderately Rigid High-Income Owner-Occupiers"
}
# Replace on all the dataframes they appear
df_clusters["cluster"] = df_clusters["cluster"].replace(cluster_names)
df_elect["cluster"] = df_elect["cluster"].replace(cluster_names)
df_household["cluster"] = df_household["cluster"].replace(cluster_names)
summary_windows["cluster"] = summary_windows["cluster"].replace(cluster_names)

In [ ]:
df1.columns

In [ ]:
# Apply the same on the expanded
df = df.merge(df_clusters, on='mainid', how='left')
df["cluster"] = df["cluster"].replace(cluster_names)

In [ ]:
# save expanded again
df.to_csv('second_resampled_tus.csv')

In [ ]:
# Save summary of rigidity measure
summary_windows.to_csv('Rigidity Measure Table.csv', index=False)

## 5.2 What activities affected their rigidity most?

In [ ]:
# Calculate per person
df_per_person = df.groupby(['mainid', 'cluster', 'activity_description'])['bin_duration_minutes'].agg(
    max_bin='max', mean_bin='mean').reset_index()

df_per_person['load_factor'] = df_per_person['mean_bin'] / df_per_person['max_bin'].replace(0, np.nan)

In [ ]:
# Then aggregate across people within each cluster
df_activity_rigidity_metrics = (df_per_person.groupby(['cluster', 'activity_description'])['load_factor']
    .agg(load_factor_mean='mean', load_factor_std='std')
    .round(3)
    .reset_index())

Save the dataframe.

In [ ]:
# Pivot by activities and index by cluster names
df_activity_rigidity = (df_activity_rigidity_metrics
    .pivot_table(index='cluster',
                 columns='activity_description',
                 values=['load_factor_mean', 'load_factor_std'],
                 aggfunc='mean'))

# Save to the activity flexibility measures
df_activity_rigidity.to_csv('activities_rigidity_measures.csv')

### 5.2.1 How is the rigidity during the daytime?

Calculate load factor for activities during the morning peak

In [ ]:
# Select the morning period
df_daytime_lf = df[
    df['starttime_of_day_min'].isin(day_engagement)]

In [ ]:
# Group by the time of the activity
df_daytime_lf = df_daytime_lf.groupby(
    ['mainid', 'cluster', 'activity_description']
)['bin_duration_minutes'].agg([
    ('max_bin', 'max'),
    ('mean_bin', 'mean')
]).reset_index()

In [ ]:
# Calculate the morning peak rigidity measure
df_daytime_lf['daytime_load_factor'] = (
    df_daytime_lf['mean_bin'] / df_daytime_lf['max_bin'].replace(0, np.nan))# change the zeros to avoid undefined during division

# replace all NaN with zero
df_daytime_lf = df_daytime_lf.fillna(0)

Save the daytime rigidity scores.

In [ ]:
# Aggregate the data
df_daytime_cluster = (
    df_daytime_lf
    .groupby(['cluster', 'activity_description'])['daytime_load_factor']
    .agg(daytime_lf_mean='mean', daytime_lf_std='std')
    .round(3)
    .reset_index()
)

In [ ]:
# Make activities become columns, with metrics as top-level columns
df_daytime_cluster = (
    df_daytime_cluster
    .pivot_table(
        index='cluster',
        columns='activity_description',
        values=['daytime_lf_mean', 'daytime_lf_std']
    )
)

In [ ]:
# Save the pivoted data
df_daytime_cluster.to_csv('activities_daytime_load_factor.csv')

### 5.2.2 How is the rigidity during the evening time?

Calculate load factor for activities during the evening peak period

In [ ]:
# Select the peak period
df_evening_lf = df[
    df['starttime_of_day_min'].isin(evening_engagement)]

In [ ]:
# Group by the time of the activity
df_evening_lf = df_evening_lf.groupby(
    ['mainid', 'cluster', 'activity_description']
)['bin_duration_minutes'].agg([
    ('max_bin', 'max'),
    ('mean_bin', 'mean')
]).reset_index()

In [ ]:
# Calculate the eveining peak flexibility measure
df_evening_lf['evening_load_factor'] = (
    df_evening_lf['mean_bin'] / df_evening_lf['max_bin'].replace(0, np.nan)) # change the zeros to avoid undefined during division

# Fill na with zero
df_evening_lf = df_evening_lf.fillna(0)

Save the evening rigidity data

In [ ]:
# Evening peak
df_evening_cluster = (df_evening_lf
    .groupby(['cluster', 'activity_description'])['evening_load_factor']
    .agg(evening_lf_mean='mean', evening_lf_std='std')
    .round(3)
    .reset_index())

In [ ]:
# Make activities become columns, with metrics as top-level columns
df_evening_cluster = (df_evening_cluster
    .pivot_table(
        index='cluster',
        columns='activity_description',
        values=['evening_lf_mean','evening_lf_std']))

In [ ]:
# Save data
df_evening_cluster.to_csv('activities_evening_load_factor.csv')

### 5.2.3 How is the reigidity during the offpeak time?

Calculate load factor for activities during off peak day

In [ ]:
# Select the peak period
df_offpeak_lf = df[
    df['starttime_of_day_min'].isin(offpeak_engagement)]

In [ ]:
# Group by the time of the activity
df_offpeak_lf = df_offpeak_lf.groupby(
    ['mainid', 'cluster', 'activity_description']
)['bin_duration_minutes'].agg([
    ('max_bin', 'max'),
    ('mean_bin', 'mean')
]).reset_index()

In [ ]:
# Calculate the off peak day flexibility measure
df_offpeak_lf['offpeak_load_factor'] = (
    df_offpeak_lf['mean_bin'] / df_offpeak_lf['max_bin'].replace(0, np.nan)) # change the zeros to avoid undefined during division

# Replace all NaN with zero after division
df_offpeak_lf = df_offpeak_lf.fillna(0)

Save the off peak rigidity measures.

In [ ]:
# Aggregate Offpeak
df_offpeak_cluster = (
    df_offpeak_lf
    .groupby(['cluster', 'activity_description'])['offpeak_load_factor']
    .agg(offpeak_lf_mean='mean', offpeak_lf_std='std')
    .round(3)
    .reset_index())

In [ ]:
# Make activities become columns, with metrics as top-level columns
df_offpeak_cluster = (df_offpeak_cluster
    .pivot_table(index='cluster',
                 columns='activity_description',
                 values=['offpeak_lf_mean','offpeak_lf_std']))

In [ ]:
# Save the dataframe
df_offpeak_cluster.to_csv('activities_offpeak_load_factor.csv')

### 5.2.4 Visualisation

I will visualise the separate periods together in comparison. I will merge them first, before plotting.

In [ ]:
# Start with the whole day rigidity measure and the daytime
df_activity_rigidity = df_activity_rigidity.merge(df_daytime_cluster, on='cluster', how='left')

# Add the evening period
df_activity_rigidity = df_activity_rigidity.merge(df_evening_cluster, on='cluster', how='left')

# Add the offpeak period
df_activity_rigidity = df_activity_rigidity.merge(df_offpeak_cluster, on='cluster', how='left')

In [ ]:
# Save the table
df_activity_rigidity.to_csv('activity_rigidity.csv')

In [ ]:
# Display the dataframe
df_activity_rigidity

In [ ]:
# Save the data
df_activity_rigidity.to_csv('Rigidity of Activities by Clusters.csv')

In [ ]:
# Extract the unique activity names from the second level of the multi-index columns
activities = df_activity_rigidity.columns.get_level_values(1).unique()

In [ ]:
# Specify the metric columns to keep from the dataset
metric_names = ['load_factor_mean', 'daytime_lf_mean', 'evening_lf_mean', 'offpeak_lf_mean']

# Define readable labels for the time periods to be displayed on the heatmap
periods = ['Overall', 'Daytime', 'Evening Peak', 'Offpeak']

In [ ]:
# Filter the dataframe to retain only the selected mean load factor metrics
df_means_only = df_activity_rigidity.loc[:,
    df_activity_rigidity.columns.get_level_values(0).isin(metric_names)
]

In [ ]:
# Index the dataframe to get the cluster column from the multiindex
clusters = df_activity_rigidity.index

In [ ]:
# Loop through each cluster to create a separate heatmap
for cluster in clusters:
    # Select the data for the current cluster and reshape it for heatmap plotting
    # so that activities form the rows and metrics form the columns
    df_heatmap = df_means_only.loc[cluster].unstack(level=0)

    # Rename the columns from technical metric names to readable period labels
    df_heatmap.columns = periods

    # Create the figure and adjust height based on the number of activities
    plt.figure(figsize=(8, max(5, len(activities) * 0.4)))

    # Plot the heatmap showing load factor values by activity and time period
    sns.heatmap(df_heatmap, annot=True, fmt='.3f', cmap='YlOrBr',
                cbar_kws={'label': 'Load Factor'}, linewidths=0.5,
                linecolor='white', vmin=0.475, vmax=1.0)

    # Add a descriptive title for the current cluster
    plt.title(f'Activities Rigidity Measure for {cluster}',
              fontweight='bold', fontsize=10)

    # Label the x-axis and y-axis
    plt.xlabel('Time Period', fontsize=8)
    plt.ylabel('Activity', fontsize=8)
    plt.tight_layout()    # Adjust layout to prevent overlap

    # Save the figure using a cleaned file name
    plt.savefig(f'Activities_flexibility_Measure_for_{cluster}.png'.replace('/', '_'),
                dpi=300,bbox_inches='tight')

    plt.show()    # Display the heatmap
    plt.close()    # Close the figure before the next loop iteration

For Moderately Rigid No‑Children Owner‑Workers, rigidity is primarily associated with laundry and dishwashing, while more discretionary activities such as watching television exhibit substantially lower rigidity. Moderately Rigid Working Families with Mortgages show rigidity concentrated in laundry and washing & dressing, reflecting routine household maintenance activities, whereas most other activities display comparatively weaker rigidity signals.

Clusters characterised by retirement show more pronounced rigidity in essential daily routines. Rigid Retired Small‑Household Owners and Rigid Low‑Income Fully Retired Owners are particularly rigid with heating‑related activities and laundry, indicating highly regularised household routines, while leisure activities remain less rigid. Similarly, Rigid Low‑Income Retired Third‑Party Housing households display their strongest rigidity in washing & dressing, suggesting highly habitual personal‑care routines.

Family‑oriented clusters exhibit rigidity across a broader range of household tasks. Moderately Rigid Upper‑Middle‑Income Large Households, Moderately Rigid Shared‑Ownership Family Households, and Moderately Rigid High‑Income Dual‑Earner Family Households show strongest rigidity in dishwashing, laundry, and food preparation, reflecting the structured demands of larger or multi‑adult households. In these clusters, rigidity in leisure‑oriented activities is consistently lower, indicating greater flexibility outside core domestic tasks.

Lower‑income working clusters demonstrate a mixed rigidity profile. Moderately Rigid Low‑Income Onsite‑Working Owners and Moderately Rigid Low‑Income Medium‑Household Workers are most rigid with dishwashing and food preparation, while activities such as watching television and, in some cases, washing & dressing, remain comparatively flexible. Moderately Rigid Low‑Income Unemployed Council Renters show heightened rigidity in house‑cleaning and laundry, suggesting constrained and repetitive household activity patterns driven by limited labour‑market engagement.

Clusters identified as less rigid overall show weaker and more selective activity rigidity. Less Rigid Lower‑Middle‑Income Onsite Workers, Less Rigid Parental‑Leave Families, and Less Rigid Lower‑Middle‑Income Retired Homeowners demonstrate limited rigidity, typically concentrated only in essential tasks such as dishwashing, with most other activities remaining highly flexible across the day.

High‑income professional clusters display selective rigidity patterns. Moderately Rigid Very‑High‑Income Remote/Onsite Professionals and Moderately Rigid Very‑High‑Income Owner‑Occupiers show rigidity primarily in food preparation and laundry, while maintaining flexibility in leisure activities. This suggests that rigidity in these clusters is driven less by time scarcity and more by habitual domestic preferences.

### Save data

In [ ]:
df_cluster_merge.to_csv('cluster_merge.csv')